In [2]:
import torch
from torch import nn
from torch.nn import functional as F

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=True, stride=1):
        super(Residual, self).__init__()
        self.seq = nn .Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        
        if use_1x1conv:
            self.res_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.res_conv = None
            
    def forward(self, X):
        Y = self.seq(X)
        
        if self.res_conv:
            X = self.res_conv(X)
            
        Y += X
        
        return F.relu(Y)

In [8]:
# 测试
blk = Residual(3, 24, use_1x1conv=True, stride=2)
blk(torch.rand(1, 3, 16, 16)).shape

torch.Size([1, 24, 8, 8])

In [21]:
blk = Residual(3, 3, use_1x1conv=False, stride=1)
blk(torch.rand(1, 3, 16, 16)).shape

torch.Size([1, 3, 16, 16])

In [22]:
blk = Residual(3, 128, use_1x1conv=True, stride=3)
blk(torch.rand(1, 3, 16, 16)).shape

torch.Size([1, 128, 6, 6])